# Renger 2026 Local Parameter Inference

**Objective**
- Back out `QB1` and `QB2` bare local priors so the parked reduced effective model reproduces the paper dressed qubit frequencies while the isolated local qubit anharmonicities stay anchored at `-0.187 GHz`.
- Refit parked local `TC1` and `TC2` priors using explicit isolated-device anchors.
- Freeze one reproducible `paper_local_priors_v1` snapshot for later coupled-model work.

**Success criteria**
- `devices.QB*.EJmax/EC/flux/asymmetry` remain bare local model inputs and `devices.QB*.f01_ghz` remain derived isolated local bare frequencies.
- The notebook reports both the paper dressed qubit references and the fitted parked reduced-model dressed qubit frequencies.
- `TC1` and `TC2` reproduce the parked isolated coupler anchors `f01 ~= 6.5 GHz` and `alpha_c ~= -0.11 GHz` at `flux = 0.0` while remaining explicitly labeled weakly identified.
- The notebook writes `output/renger2026/paper_local_priors.toml` and demonstrates uncoupled `FluxControl` sanity checks.


## Outline

1. Activate the repository environment and load the current `QuantumCircuit` API.
2. Encode the paper targets from Table II, Table I, and Appendix C.
3. Build notebook-local forward models with `CircuitHamiltonianSpec` and a compact parked reduced effective model.
4. Refit the parked couplers and back out bare qubit priors from paper dressed `f01` plus local `alpha`.
5. Inspect derived bare frequencies, dressed reduced-model residuals, and fit identifiability.
6. Freeze one accepted `paper_local_priors_v1` snapshot.
7. Run uncoupled `FluxControl` sanity checks on the inferred devices.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit
using Printf
using TOML
using QuantumToolbox: Ket, QuantumObject, dag, eigenstates
using UnicodePlots: lineplot

project_root


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


"/Users/yalgaeahn/Research/20_Projects/QuantumCircuit.jl/"

## Step 1 - Paper targets and modeling assumptions

This task stays local for device construction, but the qubit freeze now uses a compact parked reduced effective model to back out bare qubit priors from the paper dressed frequencies.

Modeling choices for this notebook:

- Use exact single-device spectra from `CircuitHamiltonianSpec(charge_cutoff = 10)` for local `f01` and `alpha`.
- Use a compact parked reduced effective model with overlap-based dressed-mode assignment for qubit back-out.
- Keep the resonator bare frequency fixed at `4.3 GHz` and record the paper dressed reference `4.22 GHz` separately; neither is inferred here.
- Treat Table II qubit `f01` values as dressed paper references and `alpha_q = -0.187 GHz` as an explicit local isolated anchor.
- Keep Table II qubit `EJ` and `EJ/EC` as paper metadata only in this pass; they are no longer hard local-prior constraints.
- Keep parked operating points at the sweet spot with `flux = 0.0`.
- Treat couplers as parked local prior refits: `f01 = 6.5 GHz` and `alpha_c ~= -0.11 GHz` are explicit isolated-device anchors, while asymmetry remains prior-held at `0.10` until coupled avoided-crossing data is added.


In [3]:
paper_targets = Dict(
    :QB1 => (
        kind = :qubit,
        name = :qb1,
        paper_dressed_f01 = 4.67,
        target_alpha = -0.187,
        paper_table2_EJ = 14.8,
        paper_table2_EJ_over_EC = 74.2,
        paper_table2_EC = 14.8 / 74.2,
        parked_flux = 0.0,
        asymmetry_prior = 0.10,
        ncut = 15,
        charge_cutoff = 10,
    ),
    :QB2 => (
        kind = :qubit,
        name = :qb2,
        paper_dressed_f01 = 4.47,
        target_alpha = -0.187,
        paper_table2_EJ = 13.8,
        paper_table2_EJ_over_EC = 68.8,
        paper_table2_EC = 13.8 / 68.8,
        parked_flux = 0.0,
        asymmetry_prior = 0.10,
        ncut = 15,
        charge_cutoff = 10,
    ),
    :TC1 => (
        kind = :coupler,
        name = :tc1,
        target_f01 = 6.5,
        target_alpha = -0.11,
        parked_flux = 0.0,
        asymmetry_prior = 0.10,
        ncut = 13,
        charge_cutoff = 10,
    ),
    :TC2 => (
        kind = :coupler,
        name = :tc2,
        target_f01 = 6.5,
        target_alpha = -0.11,
        parked_flux = 0.0,
        asymmetry_prior = 0.10,
        ncut = 13,
        charge_cutoff = 10,
    ),
)

secondary_constraints = (
    resonator_bare_f01 = 4.3,
    resonator_dressed_f01 = 4.22,
    beta_qc_qb1 = 0.017,
    beta_qc_qb2 = 0.022,
    beta_cr = 0.0225,
    beta_qr = 0.0020,
    coupler_alpha_prior = -0.11,
)

target_summary = [
    (
        device = String(label),
        kind = String(target.kind),
        paper_dressed_f01_ghz = target.kind == :qubit ? get(target, :paper_dressed_f01, missing) : missing,
        target_local_f01_ghz = target.kind == :coupler ? get(target, :target_f01, missing) : missing,
        target_alpha_ghz = get(target, :target_alpha, missing),
        paper_table2_EJ_ghz = get(target, :paper_table2_EJ, missing),
        paper_table2_EC_ghz = get(target, :paper_table2_EC, missing),
        paper_table2_EJ_over_EC = get(target, :paper_table2_EJ_over_EC, missing),
    ) for (label, target) in pairs(paper_targets)
]

sort!(target_summary, by = row -> row.device)
target_summary


4-element Vector{NamedTuple{(:device, :kind, :target_f01_ghz, :target_alpha_ghz, :target_EJ_ghz, :target_EC_ghz)}}:
 (device = "QB1", kind = "qubit", target_f01_ghz = 4.67, target_alpha_ghz = missing, target_EJ_ghz = 14.8, target_EC_ghz = 0.19946091644204852)
 (device = "QB2", kind = "qubit", target_f01_ghz = 4.47, target_alpha_ghz = missing, target_EJ_ghz = 13.8, target_EC_ghz = 0.20058139534883723)
 (device = "TC1", kind = "coupler", target_f01_ghz = 6.5, target_alpha_ghz = -0.11, target_EJ_ghz = missing, target_EC_ghz = missing)
 (device = "TC2", kind = "coupler", target_f01_ghz = 6.5, target_alpha_ghz = -0.11, target_EJ_ghz = missing, target_EC_ghz = missing)

## Step 2 - Notebook-local forward model and diagnostics

Every candidate parameter set is scored from exact single-device spectra. The diagnostic bundle includes:

- `f01`
- anharmonicity `alpha`
- parked effective Josephson energy `EJ_eff`
- numerical first-order flux sensitivity `df01/dflux`


In [4]:
effective_josephson_energy(EJmax, flux, asymmetry) =
    EJmax * sqrt(cospi(flux)^2 + (asymmetry * sinpi(flux))^2)

circuit_spec(target) = CircuitHamiltonianSpec(charge_cutoff = target.charge_cutoff)

function build_local_device(target, params)
    if target.kind == :qubit
        return TunableTransmon(
            target.name;
            EJmax = params.EJmax,
            EC = params.EC,
            flux = params.flux,
            asymmetry = params.asymmetry,
            ng = 0.0,
            ncut = target.ncut,
        )
    elseif target.kind == :coupler
        return TunableCoupler(
            target.name;
            EJmax = params.EJmax,
            EC = params.EC,
            flux = params.flux,
            asymmetry = params.asymmetry,
            ng = 0.0,
            ncut = target.ncut,
        )
    end

    error("Unsupported local target kind $(target.kind).")
end

function local_metrics(target, params; levels = 4)
    device = build_local_device(target, params)
    system = CompositeSystem(device)
    spec = spectrum(system; levels = levels, hamiltonian_spec = circuit_spec(target))
    freqs = transition_frequencies(spec)
    return (
        f01 = freqs[1],
        alpha = anharmonicity(spec),
        effEJ = effective_josephson_energy(params.EJmax, params.flux, params.asymmetry),
    )
end

function flux_slope(target, params; delta = 1e-4)
    plus = local_metrics(target, merge(params, (; flux = params.flux + delta))).f01
    minus = local_metrics(target, merge(params, (; flux = params.flux - delta))).f01
    return (plus - minus) / (2 * delta)
end

function base_params(row)
    return (
        EJmax = row.EJmax,
        EC = row.EC,
        flux = row.flux,
        asymmetry = row.asymmetry,
    )
end

local_metrics(paper_targets[:QB1], (EJmax = paper_targets[:QB1].paper_table2_EJ, EC = paper_targets[:QB1].paper_table2_EC, flux = 0.0, asymmetry = 0.10))


(f01 = 4.667382162451687, alpha = -0.22227159778090488, effEJ = 14.9)

## Step 3 - Deterministic loss functions and search strategy

The search is intentionally simple and dependency-free:

1. Qubits use a deterministic coarse grid over `flux` and `asymmetry` with an inner exact scan over `EC` and `EJmax`.
2. Couplers use a parked two-parameter solve over `EC` and `EJmax` with `flux = 0.0` and `asymmetry = 0.10` held fixed.
3. Both paths use a small exact multistart coordinate descent refinement.

The qubit loss prioritizes `f01`, derived `EC`, and a broad parked `EJ_eff` target. The coupler loss now prioritizes the explicit parked isolated-device anchors `f01 = 6.5 GHz` and `alpha_c ~= -0.11 GHz` at the sweet spot.


In [5]:
const REDUCED_BACKOUT_QUBIT_NCUT = 2
const REDUCED_BACKOUT_COUPLER_NCUT = 2
const REDUCED_BACKOUT_RESONATOR_DIM = 2

function qubit_seed_loss(target, params)
    metrics = local_metrics(target, params)

    loss = ((metrics.f01 - target.paper_dressed_f01) / 0.015)^2 +
           ((metrics.alpha - target.target_alpha) / 0.0015)^2 +
           ((params.flux - target.parked_flux) / 0.002)^2 +
           ((params.asymmetry - target.asymmetry_prior) / 0.01)^2

    return merge(params, metrics, (; loss, slope = 0.0))
end

function coupler_loss(target, params; include_slope = false)
    metrics = local_metrics(target, params)
    slope = include_slope ? flux_slope(target, params) : 0.0

    loss = ((metrics.f01 - target.target_f01) / 0.005)^2 +
           ((metrics.alpha - target.target_alpha) / 0.001)^2 +
           ((params.flux - target.parked_flux) / 0.005)^2 +
           ((params.asymmetry - target.asymmetry_prior) / 0.01)^2

    include_slope && (loss += (slope / 0.05)^2)

    return merge(params, metrics, (; loss, slope))
end

evaluate_candidate(target, params; include_slope = false) =
    target.kind == :qubit ? qubit_seed_loss(target, params) :
    coupler_loss(target, params; include_slope = include_slope)

function update_topk!(rows, candidate; k = 5)
    push!(rows, candidate)
    sort!(rows, by = row -> row.loss)
    length(rows) > k && resize!(rows, k)
    return rows
end

function parameter_bounds(target)
    if target.kind == :qubit
        return (
            EJmax = (target.paper_table2_EJ - 4.0, target.paper_table2_EJ + 4.0),
            EC = (0.145, 0.215),
            flux = (target.parked_flux, target.parked_flux),
            asymmetry = (target.asymmetry_prior, target.asymmetry_prior),
        )
    end

    return (
        EJmax = (48.0, 54.0),
        EC = (0.104, 0.110),
        flux = (target.parked_flux, target.parked_flux),
        asymmetry = (target.asymmetry_prior, target.asymmetry_prior),
    )
end

function coarse_grids(target)
    if target.kind == :qubit
        return (
            flux = [target.parked_flux],
            asymmetry = [target.asymmetry_prior],
            EC = collect(0.160:0.002:0.204),
            EJmax = collect(target.paper_table2_EJ .+ (-3.0:0.25:3.0)),
        )
    end

    return (
        flux = [target.parked_flux],
        asymmetry = [target.asymmetry_prior],
        EC = collect(0.1060:0.0005:0.1085),
        EJmax = collect(49.0:0.25:52.5),
    )
end

function set_param(params, field::Symbol, value)
    field == :EJmax && return merge(params, (; EJmax = value))
    field == :EC && return merge(params, (; EC = value))
    field == :flux && return merge(params, (; flux = value))
    field == :asymmetry && return merge(params, (; asymmetry = value))
    error("Unknown parameter field $field.")
end

function clamp_params(target, params)
    bounds = parameter_bounds(target)
    return (
        EJmax = clamp(params.EJmax, bounds.EJmax[1], bounds.EJmax[2]),
        EC = clamp(params.EC, bounds.EC[1], bounds.EC[2]),
        flux = clamp(params.flux, bounds.flux[1], bounds.flux[2]),
        asymmetry = clamp(params.asymmetry, bounds.asymmetry[1], bounds.asymmetry[2]),
    )
end


clamp_params (generic function with 1 method)

In [6]:
function inner_parameter_search(target, flux, asymmetry; keep = 4)
    grids = coarse_grids(target)
    rows = NamedTuple[]

    for EC in grids.EC
        for EJmax in grids.EJmax
            params = (EJmax = EJmax, EC = EC, flux = flux, asymmetry = asymmetry)
            candidate = evaluate_candidate(target, params; include_slope = false)
            update_topk!(rows, candidate; k = keep)
        end
    end

    return rows
end

function coordinate_descent(target, seed)
    steps = if target.kind == :qubit
        Dict(
            :EJmax => [0.20, 0.10, 0.05, 0.02],
            :EC => [0.0040, 0.0020, 0.0010, 0.0005],
        )
    else
        Dict(
            :EJmax => [0.10, 0.05, 0.02, 0.01],
            :EC => [0.0005, 0.0002, 0.0001, 0.00005],
        )
    end
    fields = (:EJmax, :EC)

    best = evaluate_candidate(target, clamp_params(target, seed); include_slope = target.kind == :coupler)

    for stage in eachindex(steps[:EJmax])
        improved = true
        while improved
            improved = false
            params = base_params(best)
            for field in fields
                for direction in (-1.0, 1.0)
                    trial_value = getfield(params, field) + direction * steps[field][stage]
                    trial_params = clamp_params(target, set_param(params, field, trial_value))
                    trial = evaluate_candidate(target, trial_params; include_slope = target.kind == :coupler)
                    if trial.loss + 1e-10 < best.loss
                        best = trial
                        improved = true
                        params = base_params(best)
                    end
                end
            end
        end
    end

    return best
end

function deduplicate_rows(rows; keep = 5)
    unique_rows = NamedTuple[]
    seen = Set{NTuple{4, Int}}()

    for row in sort(rows, by = item -> item.loss)
        key = (
            round(Int, row.EJmax * 10_000),
            round(Int, row.EC * 1_000_000),
            round(Int, row.flux * 10_000),
            round(Int, row.asymmetry * 10_000),
        )
        key in seen && continue
        push!(seen, key)
        push!(unique_rows, row)
        length(unique_rows) >= keep && break
    end

    return unique_rows
end

function run_inference(target; seeds_per_slice = 3, multistart = 8, keep = 5)
    grids = coarse_grids(target)
    seeds = NamedTuple[]

    for flux in grids.flux
        for asymmetry in grids.asymmetry
            for seed in inner_parameter_search(target, flux, asymmetry; keep = seeds_per_slice)
                update_topk!(seeds, seed; k = 20)
            end
        end
    end

    refined = NamedTuple[]
    for seed in first(seeds, min(multistart, length(seeds)))
        update_topk!(refined, coordinate_descent(target, base_params(seed)); k = keep * 3)
    end

    return deduplicate_rows(refined; keep = keep)
end

function reduced_seed_couplings(qb1_local, qb2_local, tc1_row, tc2_row)
    return (
        g_qr_qb1 = secondary_constraints.beta_qr * sqrt(qb1_local.f01 * secondary_constraints.resonator_bare_f01),
        g_qr_qb2 = secondary_constraints.beta_qr * sqrt(qb2_local.f01 * secondary_constraints.resonator_bare_f01),
        g_qc_qb1_tc1 = secondary_constraints.beta_qc_qb1 * sqrt(qb1_local.f01 * tc1_row.f01),
        g_qc_qb2_tc2 = secondary_constraints.beta_qc_qb2 * sqrt(qb2_local.f01 * tc2_row.f01),
        g_cr_tc1 = secondary_constraints.beta_cr * sqrt(tc1_row.f01 * secondary_constraints.resonator_bare_f01),
        g_cr_tc2 = secondary_constraints.beta_cr * sqrt(tc2_row.f01 * secondary_constraints.resonator_bare_f01),
    )
end

function build_reduced_backout_system(qb1_params, qb2_params, tc1_row, tc2_row)
    qb1_local = local_metrics(paper_targets[:QB1], qb1_params)
    qb2_local = local_metrics(paper_targets[:QB2], qb2_params)
    seeds = reduced_seed_couplings(qb1_local, qb2_local, tc1_row, tc2_row)

    system = CompositeSystem(
        TunableTransmon(
            :QB1;
            EJmax = qb1_params.EJmax,
            EC = qb1_params.EC,
            flux = qb1_params.flux,
            asymmetry = qb1_params.asymmetry,
            ng = 0.0,
            ncut = REDUCED_BACKOUT_QUBIT_NCUT,
        ),
        TunableCoupler(
            :TC1;
            EJmax = tc1_row.EJmax,
            EC = tc1_row.EC,
            flux = tc1_row.flux,
            asymmetry = tc1_row.asymmetry,
            ng = 0.0,
            ncut = REDUCED_BACKOUT_COUPLER_NCUT,
        ),
        Resonator(:CR; ω = secondary_constraints.resonator_bare_f01, dim = REDUCED_BACKOUT_RESONATOR_DIM),
        TunableCoupler(
            :TC2;
            EJmax = tc2_row.EJmax,
            EC = tc2_row.EC,
            flux = tc2_row.flux,
            asymmetry = tc2_row.asymmetry,
            ng = 0.0,
            ncut = REDUCED_BACKOUT_COUPLER_NCUT,
        ),
        TunableTransmon(
            :QB2;
            EJmax = qb2_params.EJmax,
            EC = qb2_params.EC,
            flux = qb2_params.flux,
            asymmetry = qb2_params.asymmetry,
            ng = 0.0,
            ncut = REDUCED_BACKOUT_QUBIT_NCUT,
        ),
        CapacitiveCoupling(:QB1, :TC1; g = seeds.g_qc_qb1_tc1),
        CapacitiveCoupling(:TC1, :CR; g = seeds.g_cr_tc1),
        CapacitiveCoupling(:CR, :TC2; g = seeds.g_cr_tc2),
        CapacitiveCoupling(:TC2, :QB2; g = seeds.g_qc_qb2_tc2),
    )

    return (; system, qb1_local, qb2_local, seeds)
end

function dressed_qubit_assignment(system)
    spec = EffectiveHamiltonianSpec()
    model = build_model(system; hamiltonian_spec = spec)
    eigensolve = eigenstates(hamiltonian(model))
    energies = Float64.(real.(eigensolve.values))
    order = sortperm(energies)
    sorted_energies = energies[order]
    sorted_vectors = eigensolve.vectors[:, order]

    qb1_bare = basis_state(system; hamiltonian_spec = spec, QB1 = 1, TC1 = 0, CR = 0, TC2 = 0, QB2 = 0)
    qb2_bare = basis_state(system; hamiltonian_spec = spec, QB1 = 0, TC1 = 0, CR = 0, TC2 = 0, QB2 = 1)
    dims = qb1_bare.dims
    dressed_states = [QuantumObject(sorted_vectors[:, idx]; type = Ket(), dims = dims) for idx in axes(sorted_vectors, 2)]
    overlaps_qb1 = Float64[abs2(dag(qb1_bare) * state) for state in dressed_states]
    overlaps_qb2 = Float64[abs2(dag(qb2_bare) * state) for state in dressed_states]

    candidate_indices = collect(2:length(sorted_energies))
    best_pair = nothing
    best_score = -Inf
    for idx1 in candidate_indices, idx2 in candidate_indices
        idx1 == idx2 && continue
        score = overlaps_qb1[idx1] + overlaps_qb2[idx2]
        if score > best_score
            best_score = score
            best_pair = (idx1, idx2)
        end
    end

    best_pair === nothing && error("Failed to assign dressed qubit modes.")
    idx1, idx2 = best_pair
    second_qb1 = maximum(overlaps_qb1[idx] for idx in candidate_indices if idx != idx1)
    second_qb2 = maximum(overlaps_qb2[idx] for idx in candidate_indices if idx != idx2)
    overlap_gap_qb1 = overlaps_qb1[idx1] - second_qb1
    overlap_gap_qb2 = overlaps_qb2[idx2] - second_qb2
    ambiguous = overlaps_qb1[idx1] < 0.20 || overlaps_qb2[idx2] < 0.20 || overlap_gap_qb1 < 1e-3 || overlap_gap_qb2 < 1e-3

    return (
        qb1_index = idx1,
        qb2_index = idx2,
        qb1_dressed_f01 = sorted_energies[idx1] - sorted_energies[1],
        qb2_dressed_f01 = sorted_energies[idx2] - sorted_energies[1],
        qb1_overlap = overlaps_qb1[idx1],
        qb2_overlap = overlaps_qb2[idx2],
        qb1_overlap_gap = overlap_gap_qb1,
        qb2_overlap_gap = overlap_gap_qb2,
        ambiguous = ambiguous,
    )
end

function qubit_backout_candidate(qb1_params, qb2_params, tc1_row, tc2_row)
    built = build_reduced_backout_system(qb1_params, qb2_params, tc1_row, tc2_row)
    assignment = dressed_qubit_assignment(built.system)

    loss = ((assignment.qb1_dressed_f01 - paper_targets[:QB1].paper_dressed_f01) / 0.002)^2 +
           ((assignment.qb2_dressed_f01 - paper_targets[:QB2].paper_dressed_f01) / 0.002)^2 +
           ((built.qb1_local.alpha - paper_targets[:QB1].target_alpha) / 0.001)^2 +
           ((built.qb2_local.alpha - paper_targets[:QB2].target_alpha) / 0.001)^2

    assignment.ambiguous && (loss += 1e6)

    return (
        loss = loss,
        qb1 = merge(qb1_params, built.qb1_local),
        qb2 = merge(qb2_params, built.qb2_local),
        assignment = assignment,
        seeds = built.seeds,
    )
end

function qubit_backout_bounds(target)
    return (
        EJmax = (target.paper_table2_EJ - 4.0, target.paper_table2_EJ + 4.0),
        EC = (0.145, 0.215),
        flux = (target.parked_flux, target.parked_flux),
        asymmetry = (target.asymmetry_prior, target.asymmetry_prior),
    )
end

function clamp_backout_params(target, params)
    bounds = qubit_backout_bounds(target)
    return (
        EJmax = clamp(params.EJmax, bounds.EJmax[1], bounds.EJmax[2]),
        EC = clamp(params.EC, bounds.EC[1], bounds.EC[2]),
        flux = clamp(params.flux, bounds.flux[1], bounds.flux[2]),
        asymmetry = clamp(params.asymmetry, bounds.asymmetry[1], bounds.asymmetry[2]),
    )
end

function joint_set_param(qb1_params, qb2_params, qubit::Symbol, field::Symbol, value)
    if qubit == :QB1
        return clamp_backout_params(paper_targets[:QB1], set_param(qb1_params, field, value)), qb2_params
    elseif qubit == :QB2
        return qb1_params, clamp_backout_params(paper_targets[:QB2], set_param(qb2_params, field, value))
    end
    error("Unsupported qubit $qubit.")
end

function coordinate_descent_qubits(seed_qb1, seed_qb2, tc1_row, tc2_row)
    steps = Dict(
        :EJmax => [0.30, 0.10, 0.05, 0.02],
        :EC => [0.0040, 0.0020, 0.0010, 0.0005],
    )
    fields = (:EJmax, :EC)

    qb1_params = clamp_backout_params(paper_targets[:QB1], seed_qb1)
    qb2_params = clamp_backout_params(paper_targets[:QB2], seed_qb2)
    best = qubit_backout_candidate(qb1_params, qb2_params, tc1_row, tc2_row)

    for stage in eachindex(steps[:EJmax])
        improved = true
        while improved
            improved = false
            current_qb1 = base_params(best.qb1)
            current_qb2 = base_params(best.qb2)
            for qubit in (:QB1, :QB2)
                for field in fields
                    for direction in (-1.0, 1.0)
                        base_value = qubit == :QB1 ? getfield(current_qb1, field) : getfield(current_qb2, field)
                        trial_qb1, trial_qb2 = joint_set_param(current_qb1, current_qb2, qubit, field, base_value + direction * steps[field][stage])
                        trial = qubit_backout_candidate(trial_qb1, trial_qb2, tc1_row, tc2_row)
                        if trial.loss + 1e-10 < best.loss
                            best = trial
                            improved = true
                            current_qb1 = base_params(best.qb1)
                            current_qb2 = base_params(best.qb2)
                        end
                    end
                end
            end
        end
    end

    return best
end

function finalize_qubit_backout_row(target, row, dressed_f01, overlap, overlap_gap)
    params = base_params(row)
    return merge(
        params,
        (
            f01 = row.f01,
            alpha = row.alpha,
            effEJ = row.effEJ,
            slope = flux_slope(target, params),
            dressed_f01 = dressed_f01,
            dressed_overlap = overlap,
            dressed_support_gap = overlap_gap,
        ),
    )
end

function run_qubit_backout(tc1_row, tc2_row; local_seed_keep = 3)
    qb1_seed_rows = run_inference(paper_targets[:QB1]; keep = local_seed_keep)
    qb2_seed_rows = run_inference(paper_targets[:QB2]; keep = local_seed_keep)

    candidates = NamedTuple[]
    for qb1_seed in qb1_seed_rows, qb2_seed in qb2_seed_rows
        candidate = coordinate_descent_qubits(base_params(qb1_seed), base_params(qb2_seed), tc1_row, tc2_row)
        update_topk!(candidates, candidate; k = 5)
    end

    best = candidates[1]
    qb1_row = finalize_qubit_backout_row(
        paper_targets[:QB1],
        best.qb1,
        best.assignment.qb1_dressed_f01,
        best.assignment.qb1_overlap,
        best.assignment.qb1_overlap_gap,
    )
    qb2_row = finalize_qubit_backout_row(
        paper_targets[:QB2],
        best.qb2,
        best.assignment.qb2_dressed_f01,
        best.assignment.qb2_overlap,
        best.assignment.qb2_overlap_gap,
    )

    validation = Dict(
        :QB1 => (
            dressed_target_f01_ghz = paper_targets[:QB1].paper_dressed_f01,
            dressed_fitted_f01_ghz = round(qb1_row.dressed_f01, digits = 5),
            dressed_residual_mhz = round(1000 * (qb1_row.dressed_f01 - paper_targets[:QB1].paper_dressed_f01), digits = 2),
            target_alpha_ghz = paper_targets[:QB1].target_alpha,
            local_alpha_ghz = round(qb1_row.alpha, digits = 5),
            alpha_residual_mhz = round(1000 * (qb1_row.alpha - paper_targets[:QB1].target_alpha), digits = 2),
            assignment_overlap = round(qb1_row.dressed_overlap, digits = 6),
            support_gap = round(qb1_row.dressed_support_gap, digits = 6),
        ),
        :QB2 => (
            dressed_target_f01_ghz = paper_targets[:QB2].paper_dressed_f01,
            dressed_fitted_f01_ghz = round(qb2_row.dressed_f01, digits = 5),
            dressed_residual_mhz = round(1000 * (qb2_row.dressed_f01 - paper_targets[:QB2].paper_dressed_f01), digits = 2),
            target_alpha_ghz = paper_targets[:QB2].target_alpha,
            local_alpha_ghz = round(qb2_row.alpha, digits = 5),
            alpha_residual_mhz = round(1000 * (qb2_row.alpha - paper_targets[:QB2].target_alpha), digits = 2),
            assignment_overlap = round(qb2_row.dressed_overlap, digits = 6),
            support_gap = round(qb2_row.dressed_support_gap, digits = 6),
        ),
    )

    return (
        qb1_seed_rows = qb1_seed_rows,
        qb2_seed_rows = qb2_seed_rows,
        best = best,
        qb1_row = qb1_row,
        qb2_row = qb2_row,
        validation = validation,
    )
end

function rounded_rows(rows)
    return [
        (
            rank = index,
            loss = round(row.loss, digits = 4),
            EJmax = round(row.EJmax, digits = 4),
            EC = round(row.EC, digits = 6),
            flux = round(row.flux, digits = 4),
            asymmetry = round(row.asymmetry, digits = 4),
            f01 = round(row.f01, digits = 5),
            alpha = round(row.alpha, digits = 5),
            dressed_f01 = hasproperty(row, :dressed_f01) ? round(row.dressed_f01, digits = 5) : missing,
            dressed_overlap = hasproperty(row, :dressed_overlap) ? round(row.dressed_overlap, digits = 6) : missing,
            effEJ = round(row.effEJ, digits = 4),
            slope = round(row.slope, digits = 6),
        ) for (index, row) in enumerate(rows)
    ]
end

function residual_report(target, row)
    if target.kind == :qubit
        return (
            paper_dressed_f01_ghz = target.paper_dressed_f01,
            fitted_dressed_f01_ghz = round(row.dressed_f01, digits = 5),
            dressed_residual_mhz = round(1000 * (row.dressed_f01 - target.paper_dressed_f01), digits = 2),
            local_bare_f01_ghz = round(row.f01, digits = 5),
            target_alpha_ghz = target.target_alpha,
            local_alpha_ghz = round(row.alpha, digits = 5),
            alpha_residual_mhz = round(1000 * (row.alpha - target.target_alpha), digits = 2),
            dressed_assignment_overlap = round(row.dressed_overlap, digits = 6),
            dressed_support_gap = round(row.dressed_support_gap, digits = 6),
            slope = round(row.slope, digits = 6),
        )
    end

    return (
        f01_residual_mhz = round(1000 * (row.f01 - target.target_f01), digits = 2),
        alpha_residual_mhz = round(1000 * (row.alpha - target.target_alpha), digits = 2),
        parked_target_f01_ghz = target.target_f01,
        parked_f01_ghz = round(row.f01, digits = 5),
        parked_flux = round(row.flux, digits = 4),
        asymmetry = round(row.asymmetry, digits = 4),
        slope = round(row.slope, digits = 6),
    )
end

function fit_classification(target, rows)
    target.kind == :qubit && return "dressed_anchor_backout"
    return "weakly identified"
end


fit_classification (generic function with 1 method)

## Step 4 - Qubit back-out against the parked reduced model

The qubit branches are no longer frozen directly from Table II `EJ/EJ/EC`. Instead, this step:

- keeps `flux = 0.0` and `asymmetry = 0.10`
- uses the parked coupler priors from the unchanged local coupler solve
- searches bare `EJmax` and `EC`
- matches the paper dressed qubit frequencies in a compact parked reduced effective model
- simultaneously matches the isolated local qubit anharmonicity target `-0.187 GHz`

The fitted qubit device fields remain bare local priors, while the dressed qubit frequencies remain reporting targets.


In [7]:
tc1_rows = run_inference(paper_targets[:TC1])
tc2_rows = run_inference(paper_targets[:TC2])

qubit_backout = run_qubit_backout(tc1_rows[1], tc2_rows[1])
qb1_rows = [merge(qubit_backout.qb1_row, (; loss = qubit_backout.best.loss))]
qb2_rows = [merge(qubit_backout.qb2_row, (; loss = qubit_backout.best.loss))]

qubit_seed_tables = Dict(
    :QB1 => rounded_rows(qubit_backout.qb1_seed_rows),
    :QB2 => rounded_rows(qubit_backout.qb2_seed_rows),
)

qubit_tables = Dict(
    :QB1 => rounded_rows(qb1_rows),
    :QB2 => rounded_rows(qb2_rows),
)

qubit_residuals = Dict(
    :QB1 => residual_report(paper_targets[:QB1], qb1_rows[1]),
    :QB2 => residual_report(paper_targets[:QB2], qb2_rows[1]),
)

qubit_classification = Dict(
    :QB1 => fit_classification(paper_targets[:QB1], qb1_rows),
    :QB2 => fit_classification(paper_targets[:QB2], qb2_rows),
)

(qubit_seed_tables, qubit_tables, qubit_residuals, qubit_backout.validation, qubit_classification)


(Dict(:QB2 => [(rank = 1, loss = 0.5643, EJmax = 13.65, EC = 0.200581, flux = 0.0, asymmetry = 0.1, f01 = 4.46979, alpha = -0.22492, effEJ = 13.65, slope = 0.0), (rank = 2, loss = 1.0308, EJmax = 13.78, EC = 0.198581, flux = 0.0, asymmetry = 0.1, f01 = 4.47072, alpha = -0.22238, effEJ = 13.78, slope = 0.0)], :QB1 => [(rank = 1, loss = 0.3775, EJmax = 14.92, EC = 0.199461, flux = 0.0, asymmetry = 0.1, f01 = 4.67066, alpha = -0.22225, effEJ = 14.92, slope = 0.0), (rank = 2, loss = 1.0115, EJmax = 14.78, EC = 0.201461, flux = 0.0, asymmetry = 0.1, f01 = 4.66981, alpha = -0.22477, effEJ = 14.78, slope = 0.0)]), Dict(:QB2 => (f01_residual_mhz = -0.21, EC_residual_mhz = 0.0, effEJ_residual_mhz = -150.0, alpha_ghz = -0.22492, slope = 0.0), :QB1 => (f01_residual_mhz = 0.66, EC_residual_mhz = 0.0, effEJ_residual_mhz = 120.0, alpha_ghz = -0.22225, slope = 0.0)), Dict(:QB2 => "identified", :QB1 => "identified"))

## Step 5 - Coupler local priors

The coupler problem is still intentionally conservative, but it now has two explicit parked isolated-device anchors: `f01 = 6.5 GHz` and `alpha_c ~= -0.11 GHz`. This notebook refits only `EC` and `EJmax` at `flux = 0.0`, with asymmetry held at the prior center `0.10`. It does not try to uniquely reconstruct the full physical SQUID tuple.


In [ ]:
coupler_tables = Dict(
    :TC1 => rounded_rows(tc1_rows),
    :TC2 => rounded_rows(tc2_rows),
)

coupler_residuals = Dict(
    :TC1 => residual_report(paper_targets[:TC1], tc1_rows[1]),
    :TC2 => residual_report(paper_targets[:TC2], tc2_rows[1]),
)

coupler_classification = Dict(
    :TC1 => fit_classification(paper_targets[:TC1], tc1_rows),
    :TC2 => fit_classification(paper_targets[:TC2], tc2_rows),
)

(coupler_tables, coupler_residuals, coupler_classification)


## Step 6 - Identifiability and profile scans

The accepted qubit point now comes from a coupled back-out objective, so the most important diagnostics are:

- dressed reduced-model residuals at the accepted parked point
- local bare `alpha` residuals
- mode-assignment overlaps for the two dressed one-excitation qubit-like states

The simple profile scans below keep the local picture explicit:

- qubits: asymmetry stays flat at the sweet spot and is still prior-held at `0.10`
- couplers: a broad `EJmax` band survives because `alpha` mostly pins `EC`, not the full SQUID parameter set


In [9]:
function profile_scan(target, row, field::Symbol, values)
    params0 = base_params(row)
    return [
        begin
            trial = evaluate_candidate(target, set_param(params0, field, value); include_slope = true)
            (; value, loss = trial.loss)
        end for value in values
    ]
end

qb1_asym_scan = profile_scan(paper_targets[:QB1], qb1_rows[1], :asymmetry, 0.0:0.01:0.20)
qb2_asym_scan = profile_scan(paper_targets[:QB2], qb2_rows[1], :asymmetry, 0.0:0.01:0.20)
tc1_ej_scan = profile_scan(paper_targets[:TC1], tc1_rows[1], :EJmax, 49.0:0.5:53.0)
tc2_ej_scan = profile_scan(paper_targets[:TC2], tc2_rows[1], :EJmax, 49.0:0.5:53.0)

profile_plots = (
    qb1_asymmetry = lineplot([row.value for row in qb1_asym_scan], [row.loss for row in qb1_asym_scan]; title = "QB1 asymmetry profile (local seed objective)", xlabel = "asymmetry", ylabel = "loss"),
    qb2_asymmetry = lineplot([row.value for row in qb2_asym_scan], [row.loss for row in qb2_asym_scan]; title = "QB2 asymmetry profile (local seed objective)", xlabel = "asymmetry", ylabel = "loss"),
    tc1_ejmax = lineplot([row.value for row in tc1_ej_scan], [row.loss for row in tc1_ej_scan]; title = "TC1 EJmax profile", xlabel = "EJmax (GHz)", ylabel = "loss"),
    tc2_ejmax = lineplot([row.value for row in tc2_ej_scan], [row.loss for row in tc2_ej_scan]; title = "TC2 EJmax profile", xlabel = "EJmax (GHz)", ylabel = "loss"),
)

qubit_backout.validation, profile_plots


(qb1_asymmetry =           ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀QB1 asymmetry profile⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀ 
          ┌────────────────────────────────────────┐ 
        2 │⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠│ 
          │⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃│ 
          │⠀⠸⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠇⠀│ 
          │⠀⠀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠎⠀⠀│ 
          │⠀⠀⠀⠸⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠎⠀⠀⠀│ 
          │⠀⠀⠀⠀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠘⢆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠃⠀⠀⠀⠀⠀│ 
   loss   │⠀⠀⠀⠀⠀⠀⠈⢢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠜⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠊⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠣⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠜⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠒⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠔⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⠢⢀⣀⠀⠀⠀⠀⣀⡠⠤⠒⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
        0 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          └────────────────────────────────────────┘ 
          ⠀

## Step 7 - Freeze `paper_local_priors_v1`

The accepted snapshot is written directly to `output/renger2026/paper_local_priors.toml`.

- `QB1` and `QB2` now carry bare back-out priors inferred from dressed parked qubit anchors plus local isolated anharmonicity.
- `TC1` and `TC2` remain intentionally identical because this local parked refit still anchors only isolated-device `f01` and `alpha`.
- `CR` keeps the split semantics `bare 4.3 GHz / dressed 4.22 GHz`.


In [ ]:
function snapshot_device(target, row, classification)
    payload = Dict{String, Any}(
        "type" => target.kind == :qubit ? "TunableTransmon" : "TunableCoupler",
        "EJmax" => round(row.EJmax, digits = 6),
        "EC" => round(row.EC, digits = 12),
        "flux" => round(row.flux, digits = 6),
        "asymmetry" => round(row.asymmetry, digits = 6),
        "ng" => 0.0,
        "ncut" => target.ncut,
        "charge_cutoff" => target.charge_cutoff,
        "fit_classification" => classification,
        "f01_ghz" => round(row.f01, digits = 5),
        "anharmonicity_ghz" => round(row.alpha, digits = 5),
    )

    if target.kind == :qubit
        payload["paper_dressed_f01_ghz"] = target.paper_dressed_f01
        payload["fitted_dressed_f01_ghz"] = round(row.dressed_f01, digits = 5)
        payload["dressed_residual_mhz"] = round(1000 * (row.dressed_f01 - target.paper_dressed_f01), digits = 2)
        payload["target_alpha_ghz"] = target.target_alpha
        payload["alpha_residual_mhz"] = round(1000 * (row.alpha - target.target_alpha), digits = 2)
        payload["dressed_assignment_overlap"] = round(row.dressed_overlap, digits = 6)
        payload["dressed_support_gap"] = round(row.dressed_support_gap, digits = 6)
    else
        payload["residual_alpha_mhz"] = round(1000 * (row.alpha - target.target_alpha), digits = 2)
    end

    return payload
end

accepted_rows = Dict(
    :QB1 => qb1_rows[1],
    :QB2 => qb2_rows[1],
    :TC1 => tc1_rows[1],
    :TC2 => tc2_rows[1],
)

accepted_snapshot = Dict{String, Any}(
    "schema" => "paper_local_priors_v1",
    "paper" => "Renger et al. (2026)",
    "source_notebook" => "output/jupyter-notebook/renger2026-parameter-inference.ipynb",
    "scope" => "Local single-device priors for QB1-TC1-CR-TC2-QB2",
    "notes" => "QB1/QB2 use a parked reduced effective-model back-out: the paper dressed qubit frequencies (4.67/4.47 GHz) and local isolated qubit anharmonicity target (-0.187 GHz) anchor bare local EJmax/EC at flux = 0.0 and asymmetry = 0.10. Their device f01_ghz values remain exact isolated local bare frequencies derived from the current TunableTransmon model. Table II qubit EJ and EJ/EC are retained as paper metadata only. TC1/TC2 use parked isolated-device anchors f01 = 6.5 GHz and alpha ~= -0.11 GHz to refit EC/EJmax at flux = 0.0, while asymmetry remains prior-held at 0.10. CR uses a bare 4.3 GHz Hamiltonian input, while the paper's dressed 4.22 GHz reference is stored separately. Explicit effective g seeds are provisional coupled-fit initializations, not device truth.",
    "targets" => Dict(
        "qb1_f01_ghz" => paper_targets[:QB1].paper_dressed_f01,
        "qb1_dressed_f01_ghz" => paper_targets[:QB1].paper_dressed_f01,
        "qb1_bare_f01_ghz" => round(accepted_rows[:QB1].f01, digits = 5),
        "qb1_alpha_ghz" => paper_targets[:QB1].target_alpha,
        "qb1_ej_ghz" => paper_targets[:QB1].paper_table2_EJ,
        "qb1_ej_over_ec" => paper_targets[:QB1].paper_table2_EJ_over_EC,
        "qb1_ec_ghz" => paper_targets[:QB1].paper_table2_EC,
        "qb2_f01_ghz" => paper_targets[:QB2].paper_dressed_f01,
        "qb2_dressed_f01_ghz" => paper_targets[:QB2].paper_dressed_f01,
        "qb2_bare_f01_ghz" => round(accepted_rows[:QB2].f01, digits = 5),
        "qb2_alpha_ghz" => paper_targets[:QB2].target_alpha,
        "qb2_ej_ghz" => paper_targets[:QB2].paper_table2_EJ,
        "qb2_ej_over_ec" => paper_targets[:QB2].paper_table2_EJ_over_EC,
        "qb2_ec_ghz" => paper_targets[:QB2].paper_table2_EC,
        "cr_f01_ghz" => secondary_constraints.resonator_bare_f01,
        "cr_dressed_f01_ghz" => secondary_constraints.resonator_dressed_f01,
        "coupler_parked_f01_ghz" => paper_targets[:TC1].target_f01,
        "coupler_alpha_prior_ghz" => secondary_constraints.coupler_alpha_prior,
        "beta_qc_qb1" => secondary_constraints.beta_qc_qb1,
        "beta_qc_qb2" => secondary_constraints.beta_qc_qb2,
        "beta_cr" => secondary_constraints.beta_cr,
        "beta_qr" => secondary_constraints.beta_qr,
        "g_qr_qb1" => round(qubit_backout.best.seeds.g_qr_qb1, digits = 12),
        "g_qr_qb2" => round(qubit_backout.best.seeds.g_qr_qb2, digits = 12),
        "g_qc_qb1_tc1" => round(qubit_backout.best.seeds.g_qc_qb1_tc1, digits = 12),
        "g_qc_qb2_tc2" => round(qubit_backout.best.seeds.g_qc_qb2_tc2, digits = 12),
        "g_cr_tc1" => round(qubit_backout.best.seeds.g_cr_tc1, digits = 12),
        "g_cr_tc2" => round(qubit_backout.best.seeds.g_cr_tc2, digits = 12),
    ),
    "inference" => Dict(
        "hamiltonian_spec" => "CircuitHamiltonianSpec(charge_cutoff = 10)",
        "sweet_spot_flux_target" => 0.0,
        "asymmetry_prior_center" => 0.10,
        "qubit_fit_classification" => "dressed_anchor_backout",
        "coupler_fit_classification" => "weakly identified",
        "qubit_backout_model" => "parked reduced effective model with overlap-based dressed-mode assignment",
    ),
    "devices" => Dict(
        "QB1" => snapshot_device(paper_targets[:QB1], accepted_rows[:QB1], qubit_classification[:QB1]),
        "QB2" => snapshot_device(paper_targets[:QB2], accepted_rows[:QB2], qubit_classification[:QB2]),
        "TC1" => snapshot_device(paper_targets[:TC1], accepted_rows[:TC1], coupler_classification[:TC1]),
        "TC2" => snapshot_device(paper_targets[:TC2], accepted_rows[:TC2], coupler_classification[:TC2]),
    ),
)

output_dir = joinpath(project_root, "output", "renger2026")
mkpath(output_dir)
snapshot_path = joinpath(output_dir, "paper_local_priors.toml")
open(snapshot_path, "w") do io
    TOML.print(io, accepted_snapshot)
end

(snapshot_path, accepted_snapshot["devices"])


## Step 8 - Flux-control sanity checks on uncoupled devices

These checks stay inside the current supported surface:

- one local tunable subsystem at a time
- circuit Hamiltonian only
- no coupled-device pulse calibration yet

The two tests here are minimal but useful:

- a constant `FluxControl` shift should match the corresponding static shifted model
- a small sinusoidal pulse should produce nontrivial but stable population transfer


In [11]:
kwarg(symbol::Symbol, value) = NamedTuple{(symbol,)}((value,))

function flux_sanity_check(label, target, row; delta_flux = 0.015, pulse_delta = 0.025, pulse_omega = 4.5)
    params = base_params(row)
    device = build_local_device(target, params)
    system = CompositeSystem(device)
    spec = circuit_spec(target)
    ψ0 = basis_state(system; hamiltonian_spec = spec, kwarg(target.name, 0)...)
    t_short = collect(range(0.0, 4.0; length = 41))
    t_long = collect(range(0.0, 12.0; length = 121))
    charge_obs = ObservableSpec(Symbol(:charge_, target.name), target.name, :charge)

    shifted_system = with_subsystem_parameter(system, target.name, :flux, params.flux + delta_flux)
    shifted_result = evolve(
        shifted_system,
        basis_state(shifted_system; hamiltonian_spec = spec, kwarg(target.name, 0)...),
        t_short;
        hamiltonian_spec = spec,
        observables = [charge_obs],
    )

    constant_result = evolve(
        system,
        ψ0,
        t_short;
        hamiltonian_spec = spec,
        observables = [charge_obs],
        flux_controls = [FluxControl(Symbol(:constant_, target.name), target.name, (p, t) -> delta_flux)],
    )

    shifted_trace = observable_trace(shifted_result, charge_obs.label).values
    constant_trace = observable_trace(constant_result, charge_obs.label).values
    max_constant_error = maximum(abs.(constant_trace .- shifted_trace))

    pulse = FluxControl(
        Symbol(:pulse_, target.name),
        target.name,
        (p, t) -> p.delta * sin(p.omega * t);
        derivative = (p, t) -> p.delta * p.omega * cos(p.omega * t),
    )
    pulsed_result = evolve(
        system,
        ψ0,
        t_long;
        hamiltonian_spec = spec,
        flux_controls = [pulse],
        params = (; delta = pulse_delta, omega = pulse_omega),
    )
    excited_population = population_trace(pulsed_result, target.name, 1)

    return (
        device = String(label),
        max_constant_shift_error = max_constant_error,
        max_excited_population = maximum(excited_population.values),
        final_excited_population = excited_population.values[end],
    )
end

sanity_summary = [
    flux_sanity_check(:QB1, paper_targets[:QB1], accepted_rows[:QB1]),
    flux_sanity_check(:TC1, paper_targets[:TC1], accepted_rows[:TC1]),
]

sanity_summary


2-element Vector{@NamedTuple{device::String, max_constant_shift_error::Float64, max_excited_population::Float64, final_excited_population::Float64}}:
 (device = "QB1", max_constant_shift_error = 1.1456555538913449e-5, max_excited_population = 0.5334158311181706, final_excited_population = 0.04498041682939048)
 (device = "TC1", max_constant_shift_error = 3.864584450674613e-6, max_excited_population = 0.4143046603429137, final_excited_population = 0.08662523984456122)

## Result

This notebook produces a reproducible local prior set that matches the paper targets as closely as the single-device information allows today.

Practical conclusion:

- `QB1` and `QB2` can be frozen now as local tunable-subsystem priors.
- `TC1` and `TC2` should still be treated as initialization points, not device truth.
- The next fitting stage should use coupled avoided crossings and gate-operating-point data to separate the two coupler branches.
